# Notebook 04 — Evaluation

Compares four system configurations on a held-out test set of 20 questions:

| Config | Model | RAG |
|--------|-------|-----|
| **A1 — llama / no RAG** | llama3.2:3b | off |
| **B1 — llama / RAG** | llama3.2:3b | on (top-5) |
| **A2 — gemma / no RAG** | gemma3:4b | off |
| **B2 — gemma / RAG** | gemma3:4b | on (top-5) |

**Metrics**
- **ROUGE-L** — longest common subsequence overlap with reference answer (in-scope only)
- **Faithfulness** — word overlap between the answer and the retrieved chunks
- **Refusal rate** — % of out-of-scope questions correctly declined
- **Retrieval hit-rate** — % of in-scope questions where retrieved chunks include the expected source
- **Response time** — wall-clock ms per query

**Run order:** cells are split by model so a gemma timeout does not lose llama results.
Each model saves to its own partial file; the merge cell combines them.

**Prerequisites:** notebook 02 (ChromaDB built) and Ollama running (`ollama serve`) with both models pulled.

In [10]:
%pip install rouge-score pandas --quiet
from sentence_transformers import SentenceTransformer, util as st_util
_SIM_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
import sys
import time
import json
import subprocess
from pathlib import Path

import pandas as pd
from rouge_score import rouge_scorer

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.rag import retrieve, build_prompt, call_ollama, check_ollama
from app import settings

TOP_K       = settings.TOP_K
TEMPERATURE = settings.TEMPERATURE

# Per-model timeouts (gemma needs more time to load)
MODEL_TIMEOUT = {
    "llama3.2:3b": 120,
    "gemma3:4b":   240,
}

RESULTS_DIR = Path("../data/evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"TOP_K={TOP_K}, TEMPERATURE={TEMPERATURE}")
print(f"Results → {RESULTS_DIR.resolve()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TOP_K=5, TEMPERATURE=0.1
Results → C:\Users\boroh\ELTE\Thesis\elte_chat\data\evaluation


In [3]:
assert check_ollama(), "Ollama is not running — start with: ollama serve"

pulled = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
for m in ["llama3.2:3b", "gemma3:4b"]:
    base   = m.split(":")[0]
    status = "OK" if base in pulled else "MISSING — run: ollama pull " + m
    print(f"  {m}: {status}")

print(f"ChromaDB collection: {settings.COLLECTION_NAME}")
print("Pre-flight done.")

  llama3.2:3b: OK
  gemma3:4b: OK
ChromaDB collection: elte_ik
Pre-flight done.


In [6]:
TEST_SET = [
    # Prerequisites (Weakening_the_prerequisite_s_.pdf / prerequisites.html)
    {"id": 1,  "question": "What is a strong prerequisite?",
     "reference": "Strong prerequisites are prerequisites without which the follow-up subject cannot be taken in the next semester, and the follow-up subject can only be registered for after the prerequisite subject has been fulfilled. Neptun will automatically deregister you from the subject if you do not meet this prerequisite.",
     "in_scope": True, "expected_source": "prerequisite"},
    {"id": 2,  "question": "What is a weak prerequisite?",
     "reference": "A weak prerequisite can be taken in the same semester as the follow-up subject, but you must complete it first before you pass the follow-up. If not completed, your grade in the follow-up subject will automatically be failed or unfulfilled even if you pass its exam.",
     "in_scope": True, "expected_source": "prerequisite"},
    {"id": 3,  "question": "If I fail a weak prerequisite, does it count toward my subject registration limit?",
     "reference": "If you register for the follow-up subject but cannot pass it in a given semester due to a failed prerequisite, the follow-up subject will not be counted into the maximum 3 subject registrations per subject limit during your studies.",
     "in_scope": True, "expected_source": "prerequisite"},

    # Academic Regulations (ELTE Academic Regulations.pdf)
    {"id": 4,  "question": "What is a passive semester at ELTE?",
     "reference": "Passive semester: the semester in which the Student announces that he/she wishes to interrupt his/her studies, or he/she cancels his/her active registration, or fails to register until the given deadline (in accordance with the Nftv, the student status is terminated if the Student fails to register for the next semester following two consecutive semesters of suspension);",
     "in_scope": True, "expected_source": "Academic Regulations"},
    {"id": 5,  "question": "What does active semester mean at ELTE?",
     "reference": "Active semester: the semester in which the Student registers to start or continue his/her studies and does not cancel it over the course of the semester;",
     "in_scope": True, "expected_source": "Academic Regulations"},
    {"id": 6,  "question": "What financial support is available for ELTE students?",
     "reference": "From the Institute’s funds provided as normative funding per student, the financial support made available to students takes the following bursary and scholarship forms [disbursement titles]: (1) Academic scholarship as a scholarship awarded on the basis of academic performance pursuant to Nftv. Section 85/C. aa). (2) 479 National Higher Education Scholarship as a scholarship awarded on the basis of academic performance pursuant to Nftv. Section 85/C. ab). (3) The following scholarships as scholarships awarded on the basis of academic performance pursuant to Nftv. Section 85/C. ac). a) scientific scholarship, b) scholarships for participation in academic competitions and conferences c) public service scholarship, d) scholarship for sports e) scholarship for cultural activities, f) compensation for parallel studies. g) 480ERASMUS+ Start h) “Good student, good athlete” scholarship i) professional scholarship k) 481 arts scholarship (4) Bursaries based social background pursuant to scholarship detailed in Nftv. Section 85/C. b) a) regular social grant, b) regular social grant 10%, c) regular social grant 20%, d) one-time social grant, e) the institutional part of Bursa Hungarica, f) normative grants for foreign students, g) basic financial support, h) financial support for traineeships; (5) Doctorate bursary pursuant to Nftv. Section 85/C. c). (6) Other scholarships awarded pursuant to Nftv. Section 85/C. d, out of the Institution’s income.",
     "in_scope": True, "expected_source": "Academic Regulations"},
    {"id": 7,  "question": "What is the result of the final examination at ELTE?",
     "reference": "The result of the final examination is the average of the grades given for the elements of the final examination defined in Section 81 (1). Any deferring regulation in connection with the grading of the final examination is (as part of the training programme) approved by the Faculty Council on the proposal of the person responsible for the degree course",
     "in_scope": True, "expected_source": "Academic Regulations"},

    # Student Services
    {"id": 8,  "question": "What is the student card and how do I get one?",
     "reference": "The student card is a plastic orange-brown card, in size similar to a bank card. You are eligible if you are staying at ELTE longer than 12 months, your semester status is active, and your Education ID has appeared in Neptun. To apply, obtain a NEK document from a Kormanyablak office, upload it in Neptun, and register your application electronically in the Neptun system.",
     "in_scope": True, "expected_source": "Student Card"},
    {"id": 9,  "question": "What housing options are available for ELTE students?",
     "reference": "The ELTE Dormitory Centre provides accommodation to hundreds of ELTE international students in two cities, Budapest, and Szombathely. The dormitories are popular among both the Hungarian and the international students as they offer not only high-quality and affordable places and services – more than half of the dormitories have been completely rebuilt within a few years –, but because of the vivid community life as well. Should you need a place in one of ELTE’s dormitories, please visit: https://www.elte.hu/en/dormitory-centre and contact them.",
     "in_scope": True, "expected_source": "housing"},
    {"id": 10, "question": "What should I do in case of a medical emergency in Hungary?",
     "reference": "Important numbers for emergency General emergency number      112 Ambulance 104 Fire 105 Police 107",
     "in_scope": True, "expected_source": "health-insurance"},

    {"id": 11, "question": "What is the Stipendium Hungaricum scholarship?",
     "reference": "Stipendium Hungaricum is a scholarship programme for foreign students, founded by the Hungarian Government in 2013. The programme aims to promote cultural understanding, economic and cultural relations between Hungary and other countries. The Department of Erasmus+ and International Programmes is responsible for the implementation of the programme at ELTE.",
     "in_scope": True, "expected_source": "stipendium"},

    # Exams and Credits
    {"id": 12, "question": "What are the rules for exam registration in Neptun?",
     "reference": "You must first register for the subject in Neptun to be eligible for the exam. Maximum 3 exam attempts per subject are allowed in the same exam period. You can register for an exam and cancel or change exam dates at least 24 hours before the exam starts. Absence without a valid reason is counted as an unsuccessful attempt.",
     "in_scope": True, "expected_source": "Exam period"},
    {"id": 13, "question": "How many credits are needed to complete a BSc at ELTE?",
     "reference": "In the BSc programme you have to complete 180 credits: compulsory subjects 127 credits, compulsory elective subjects 23 credits, elective subjects 10 credits, and thesis consultation 20 credits.",
     "in_scope": True, "expected_source": "curriculum"},

    # International
    {"id": 14, "question": "Can a Stipendium Hungaricum student go on Erasmus Scholarship?",
     "reference": "Stipendium Hungaricum students on level BSc and MSc are not eligible for Erasmus scholarship.",
     "in_scope": True, "expected_source": "info-for-outgoing-erasmus-students"},
    {"id": 15, "question": "What health insurance are ELTE students entitled to?",
     "reference": "When you arrive in Hungary, make sure that you have a health insurance with you that you can use for health care services during your stay in Hungary. Read carefully which health insurance (non-private or private) you can use for the health services in Hungary. If you are a non-EEA student (coming from outside of the European Economic Area),  you should consider which option suits your situation the best and what are you entitled to have. If you are an EEA student, it is advisable to apply for a European Health Insurance Card from you health insurance institution in your homeland before you leave from home. Please have in mind that if you are Stipendium Hungaricum, Hungarian Diaspora Scholarship or a Scholarship Programme for Christian Young People scholarship holder, you are eligible for both TAJ card (non-private health insurance) and supplementary (private) health insurance. After enrollment please start arranging them both! Please visit this page for information about non-private insurance as a scholarship holder. You can learn about health care services available for Stipendium Hungaricum, Diaspora and SCYP students here.",
     "in_scope": True, "expected_source": "health-insurance"},

    # Out-of-scope
    {"id": 16, "question": "What is the weather like in Budapest?",   "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 17, "question": "Who won the FIFA World Cup in 2022?",     "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 18, "question": "How do I cook pasta?",                    "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 19, "question": "What is the population of Hungary?",      "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 20, "question": "Can you recommend a movie to watch?",     "reference": "", "in_scope": False, "expected_source": ""},
]

in_scope  = [q for q in TEST_SET if q["in_scope"]]
out_scope = [q for q in TEST_SET if not q["in_scope"]]
print(f"Test set: {len(TEST_SET)} questions ({len(in_scope)} in-scope, {len(out_scope)} out-of-scope)")

unfilled = [q["id"] for q in in_scope if q["reference"].startswith("FILL")]
if unfilled:
    print(f"WARNING: references still need filling for question IDs: {unfilled}")


Test set: 20 questions (15 in-scope, 5 out-of-scope)


In [21]:
def build_prompt_baseline(query: str) -> str:
    return (
        "You are a helpful assistant for ELTE Faculty of Informatics students. "
        f"Answer the following question as accurately as possible.\n\nQuestion: {query}\nAnswer:"
    )


def semantic_similarity(reference: str, answer: str):
    if not reference or not answer or answer.startswith("ERROR"):
        return None
    emb_ref = _SIM_MODEL.encode(reference, convert_to_tensor=True)
    emb_ans = _SIM_MODEL.encode(answer, convert_to_tensor=True)
    return round(float(st_util.cos_sim(emb_ref, emb_ans)), 4)


REFUSAL_PHRASES = [
    # Original
    "not in the context", "i don't have", "i do not have", "not available",
    "cannot answer", "no information", "outside the scope", "i'm unable",
    "i am unable", "don't know", "do not know", "not provided",
    "only help with elte", "only answer elte", "cannot help with",
    # Added: catches Gemma-style refusals missed in v1
    "context does not contain", "context does not provide",
    "context does not include", "not mentioned in the context",
    "no information in the context", "provided context does not",
    "based on the provided context, i cannot", "the context does not",
    "only answer questions related to",
]

def is_refusal(answer: str) -> bool:
    a = answer.lower()
    return any(phrase in a for phrase in REFUSAL_PHRASES)


def retrieval_hit(chunks: list, expected_source: str) -> bool:
    """Return True if any retrieved chunk's source path contains expected_source.
    Checks source_relative (full relative path) first, falls back to file_name.
    """
    if not expected_source:
        return False
    src = expected_source.lower()
    return any(
        src in c["metadata"].get("source_relative", "").lower()
        or src in c["metadata"].get("file_name", "").lower()
        for c in chunks
    )


def run_model(model: str, chunks_cache: dict) -> list:
    """Run both RAG and no-RAG configs for one model. Returns list of result rows."""
    import requests as _req
    timeout = MODEL_TIMEOUT.get(model, 180)
    rows = []
    for use_rag in [False, True]:
        tag = f"{'B' if use_rag else 'A'}_{model.split(':')[0].replace('.', '')}"
        print(f"\n=== {tag} (model={model}, rag={use_rag}, timeout={timeout}s) ===")
        for q in TEST_SET:
            chunks = chunks_cache[q["id"]]
            prompt = build_prompt(q["question"], chunks) if use_rag else build_prompt_baseline(q["question"])
            t0 = time.monotonic()
            try:
                payload = {
                    "model": model, "prompt": prompt, "stream": False,
                    "options": {"temperature": TEMPERATURE, "num_ctx": 2048},
                }
                r = _req.post(settings.OLLAMA_URL, json=payload, timeout=timeout)
                r.raise_for_status()
                answer = r.json()["response"].strip()
            except Exception as exc:
                answer = f"ERROR: {exc}"
            ms = int((time.monotonic() - t0) * 1000)
            rows.append({
                "config": tag, "model": model, "rag": use_rag,
                "id": q["id"], "question": q["question"],
                "reference": q["reference"], "in_scope": q["in_scope"],
                "expected_source": q["expected_source"],
                "answer": answer,
                "chunks_used": [c["content"] for c in chunks],
                "response_ms": ms,
            })
            status = "ERR" if answer.startswith("ERROR") else f"{ms}ms"
            print(f"  [{q['id']:02d}] {status} — {answer[:60].replace(chr(10), ' ')}...")
    return rows


print("Helpers defined.")


Helpers defined.


In [8]:
# Build chunk cache once — shared across all configs
chunks_cache = {q["id"]: retrieve(q["question"], n_results=TOP_K) for q in TEST_SET}
print(f"Chunk cache built for {len(chunks_cache)} questions.")

Chunk cache built for 20 questions.


## Step 1 — Run llama3.2:3b (A1 + B1)
Results saved to `data/evaluation/partial_llama.json` after completion.

In [9]:
llama_rows = run_model("llama3.2:3b", chunks_cache)

partial_llama = RESULTS_DIR / "partial_llama.json"
partial_llama.write_text(json.dumps(llama_rows, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(llama_rows)} llama rows → {partial_llama}")


=== A_llama32 (model=llama3.2:3b, rag=False, timeout=120s) ===
  [01] 38952ms — A strong prerequisite is a course or requirement that is ess...
  [02] 29591ms — A weak prerequisite, also known as a "weakly required" or "r...
  [03] 26703ms — In most cases, failing a weak prerequisite will not count to...
  [04] 27684ms — At ELTE (Eötvös Loránd University), a "passive semester" ref...
  [05] 34036ms — At ELTE (Eötvös Loránd University), "active semester" refers...
  [06] 63847ms — As an assistant for ELTE Faculty of Informatics students, I ...
  [07] 40948ms — At the ELTE (Eötvös Loránd University) Faculty of Informatic...
  [08] 56842ms — At ELTE (Eötvös Loránd University), the student card, also k...
  [09] 44078ms — ELTE (Eötvös Loránd University) offers various housing optio...
  [10] 45323ms — In case of a medical emergency in Hungary, here are some ste...
  [11] 45669ms — The Stipendium Hungaricum Scholarship (SHS) is a prestigious...
  [12] 64125ms — A very specific question!  I

## Step 2 — Run gemma3:4b (A2 + B2)
Timeout is set to 240 s per call. Results saved to `data/evaluation/partial_gemma.json` after completion.

In [11]:
gemma_rows = run_model("gemma3:4b", chunks_cache)

partial_gemma = RESULTS_DIR / "partial_gemma.json"
partial_gemma.write_text(json.dumps(gemma_rows, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(gemma_rows)} gemma rows → {partial_gemma}")


=== A_gemma3 (model=gemma3:4b, rag=False, timeout=240s) ===
  [01] 166485ms — Okay, let’s talk about strong prerequisites in the context o...
  [02] 137115ms — Okay, let’s break down what a weak prerequisite is, particul...
  [03] 113167ms — Okay, let’s address this question specifically for ELTE Facu...
  [04] 150076ms — Okay, let’s break down the “passive semester” (nyári félélet...
  [05] 138905ms — Okay, let’s break down what “active semester” means at the E...
  [06] ERR — ERROR: HTTPConnectionPool(host='localhost', port=11434): Rea...
  [07] 180481ms — Okay, let’s break down the final examination system at the E...
  [08] 144917ms — Okay, let’s get you the information you need about the ELTE ...
  [09] ERR — ERROR: HTTPConnectionPool(host='localhost', port=11434): Rea...
  [10] ERR — ERROR: HTTPConnectionPool(host='localhost', port=11434): Rea...
  [11] 198206ms — Okay, here’s a detailed explanation of the Stipendium Hungar...
  [12] 201755ms — Okay, here’s a breakdown of the ex

## Step 3 — Merge results
Loads from partial files so this cell works even if you ran the model cells in separate sessions.

In [12]:
results = []
for path in [RESULTS_DIR / "partial_llama.json", RESULTS_DIR / "partial_gemma.json"]:
    if path.exists():
        results.extend(json.loads(path.read_text(encoding="utf-8")))
    else:
        print(f"WARNING: {path.name} not found — run the corresponding cell first")

raw_path = RESULTS_DIR / "raw_results.json"
raw_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Merged {len(results)} rows → {raw_path}")

configs_present = sorted({r['config'] for r in results})
print(f"Configs present: {configs_present}")

Merged 80 rows → ..\data\evaluation\raw_results.json
Configs present: ['A_gemma3', 'A_llama32', 'B_gemma3', 'B_llama32']


In [13]:
# ROUGE-L — in-scope questions only
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

for row in results:
    if row["in_scope"] and row["reference"] and not row["answer"].startswith("ERROR"):
        score = scorer.score(row["reference"], row["answer"])
        row["rouge_l"] = round(score["rougeL"].fmeasure, 4)
    else:
        row["rouge_l"] = None

df = pd.DataFrame(results)
rouge_summary = (
    df[df["rouge_l"].notna()]
    .groupby(["model", "rag"])["rouge_l"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("ROUGE-L (in-scope questions only)")
print(rouge_summary)

ROUGE-L (in-scope questions only)
                     mean     min     max
model       rag                          
gemma3:4b   False  0.0759  0.0411  0.1171
            True   0.3566  0.1217  0.7692
llama3.2:3b False  0.1278  0.0383  0.2410
            True   0.3186  0.0830  0.7470


In [16]:
# Semantic similarity — cosine distance between reference and generated answer embeddings
# Uses all-MiniLM-L6-v2, same model as retrieval. In-scope questions only.
for row in results:
    if row["in_scope"] and row["reference"] and not row["answer"].startswith("ERROR"):
        row["sem_sim"] = semantic_similarity(row["reference"], row["answer"])
    else:
        row["sem_sim"] = None

df = pd.DataFrame(results)
sim_summary = (
    df[df["sem_sim"].notna()]
    .groupby(["model", "rag"])["sem_sim"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("Semantic Similarity / cosine (in-scope questions only)")
print(sim_summary)

Semantic Similarity / cosine (in-scope questions only)
                     mean     min     max
model       rag                          
gemma3:4b   False  0.6175  0.4417  0.7952
            True   0.7779  0.5087  0.9713
llama3.2:3b False  0.6310  0.3334  0.7751
            True   0.7571  0.3139  0.9565


In [22]:
# Refusal rate — out-of-scope only
for row in results:
    row["refused_auto"]   = is_refusal(row["answer"]) if not row["in_scope"] else None
    row["refused_manual"] = None  # fill manually after review

df  = pd.DataFrame(results)
oos = df[df["in_scope"] == False]

refusal_summary = (
    oos.groupby(["model", "rag"])["refused_auto"]
    .agg(refused="sum", total="count")
    .assign(refusal_rate=lambda x: (x["refused"] / x["total"]).round(4))
)
print("Refusal rate (out-of-scope, heuristic)")
print(refusal_summary)

print("\nPer-question refusal check:")
for _, r in oos.iterrows():
    flag = "REFUSED" if r["refused_auto"] else "ANSWERED"
    print(f"  [{r['config']}] Q{r['id']}: {flag} — {r['answer'][:80].replace(chr(10), ' ')}...")

Refusal rate (out-of-scope, heuristic)
                  refused  total refusal_rate
model       rag                              
gemma3:4b   False       0      5          0.0
            True        5      5          1.0
llama3.2:3b False       0      5          0.0
            True        4      5          0.8

Per-question refusal check:
  [A_llama32] Q16: ANSWERED — I'd be happy to help you with the current weather in Budapest!  Since I'm an AI ...
  [A_llama32] Q17: ANSWERED — The winner of the 2022 FIFA World Cup was Argentina, led by Lionel Messi. They d...
  [A_llama32] Q18: ANSWERED — Cooking pasta is a straightforward process that requires some basic steps. Here'...
  [A_llama32] Q19: ANSWERED — As of my knowledge cutoff in 2023, the estimated population of Hungary is approx...
  [A_llama32] Q20: ANSWERED — I'd be happy to help with some movie recommendations!  As an assistant, I can su...
  [B_llama32] Q16: REFUSED — I can only answer questions related to the ELTE Faculty o

In [18]:
# Retrieval hit-rate — same chunks for all configs, computed once
for row in results:
    if row["in_scope"]:
        row["retrieval_hit"] = retrieval_hit(chunks_cache[row["id"]], row["expected_source"])
    else:
        row["retrieval_hit"] = None

df = pd.DataFrame(results)
hit_per_q = (
    df[df["retrieval_hit"].notna()]
    .drop_duplicates(subset=["id"])
    .assign(hit=lambda x: x["retrieval_hit"].astype(int))
)
overall_hit = hit_per_q["hit"].mean()
print(f"Overall retrieval hit-rate: {overall_hit:.2%} ({int(hit_per_q['hit'].sum())}/{len(hit_per_q)} questions)")
for _, r in hit_per_q.iterrows():
    print(f"  Q{int(r['id'])}: {'HIT' if r['hit'] else 'MISS'} (expected: {r['expected_source']})")

Overall retrieval hit-rate: 93.33% (14/15 questions)
  Q1: HIT (expected: prerequisite)
  Q2: HIT (expected: prerequisite)
  Q3: HIT (expected: prerequisite)
  Q4: HIT (expected: Academic Regulations)
  Q5: HIT (expected: Academic Regulations)
  Q6: HIT (expected: Academic Regulations)
  Q7: HIT (expected: Academic Regulations)
  Q8: HIT (expected: Student Card)
  Q9: HIT (expected: housing)
  Q10: HIT (expected: health-insurance)
  Q11: HIT (expected: stipendium)
  Q12: HIT (expected: Exam period)
  Q13: HIT (expected: curriculum)
  Q14: MISS (expected: info-for-outgoing-erasmus-students)
  Q15: HIT (expected: health-insurance)


In [19]:
df = pd.DataFrame(results)
time_summary = (
    df.groupby(["model", "rag"])["response_ms"]
    .agg(["mean", "median", "min", "max"])
    .round(0)
    .astype(int)
)
print("Response time (ms)")
print(time_summary)

Response time (ms)
                     mean  median    min     max
model       rag                                 
gemma3:4b   False  157164  158280  10742  242530
            True    66918   66318  44190  102202
llama3.2:3b False   41507   42513  13094   86446
            True    53233   51678  23971   88475


In [24]:
# Final summary table
df = pd.DataFrame(results)
df["rouge_l"]       = [r.get("rouge_l")       for r in results]
df["sem_sim"] = [r.get("sem_sim") for r in results]
df["refused_auto"]  = [r.get("refused_auto")  for r in results]
df["retrieval_hit"] = [r.get("retrieval_hit") for r in results]

hit_rates = {}
for qid, chunks in chunks_cache.items():
    q = next(q for q in TEST_SET if q["id"] == qid)
    if q["in_scope"]:
        hit_rates[qid] = retrieval_hit(chunks, q["expected_source"])
global_hit_rate = sum(hit_rates.values()) / len(hit_rates) if hit_rates else 0

summary_rows = []
for model in ["llama3.2:3b", "gemma3:4b"]:
    for use_rag in [False, True]:
        sub  = df[(df["model"] == model) & (df["rag"] == use_rag)]
        if sub.empty:
            continue
        oos  = sub[sub["in_scope"] == False]
        insc = sub[sub["in_scope"] == True]
        summary_rows.append({
            "Model":              model,
            "RAG":                use_rag,
            "ROUGE-L (avg)":      round(insc["rouge_l"].mean(), 4),
            "Sem Sim (avg)":      round(insc["sem_sim"].mean(), 4),
            "Refusal rate":       round(oos["refused_auto"].mean(), 4) if len(oos) else float("nan"),
            "Retrieval hit-rate": round(global_hit_rate, 4),
            "Avg time (ms)":      int(sub["response_ms"].mean()),
        })

summary_df = pd.DataFrame(summary_rows)
print("\n=== EVALUATION SUMMARY ===")
print(summary_df.to_string(index=False))

summary_path = RESULTS_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")
print(f"\nSaved → {summary_path}")

full_path = RESULTS_DIR / "full_results.csv"
df.drop(columns=["chunks_used"]).to_csv(full_path, index=False, encoding="utf-8")
print(f"Saved → {full_path}")

raw_path = RESULTS_DIR / "raw_results.json"
raw_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved → {raw_path}")


=== EVALUATION SUMMARY ===
      Model   RAG  ROUGE-L (avg)  Sem Sim (avg)  Refusal rate  Retrieval hit-rate  Avg time (ms)
llama3.2:3b False         0.1278         0.6310           0.0              0.9333          41506
llama3.2:3b  True         0.3186         0.7571           0.8              0.9333          53233
  gemma3:4b False         0.0759         0.6175           0.0              0.9333         157163
  gemma3:4b  True         0.3566         0.7779           1.0              0.9333          66917

Saved → ..\data\evaluation\summary.csv
Saved → ..\data\evaluation\full_results.csv
Saved → ..\data\evaluation\raw_results.json


## Manual review — out-of-scope refusals

Open `data/evaluation/full_results.csv`, filter `in_scope == False`, and fill the `refused_manual` column
(1 = correctly refused, 0 = hallucinated an answer). The phrase-list misses phrasings like
"I can only help with ELTE-related questions" — manual labelling captures these.

Re-run the cell below once labelled.

In [ ]:
full_path  = RESULTS_DIR / "full_results.csv"
df_manual  = pd.read_csv(full_path)
oos_manual = df_manual[df_manual["in_scope"] == False]

if oos_manual["refused_manual"].notna().any():
    manual_summary = (
        oos_manual.groupby(["model", "rag"])["refused_manual"]
        .agg(refused="sum", total="count")
        .assign(refusal_rate=lambda x: (x["refused"] / x["total"]).round(4))
    )
    print("Refusal rate (manual labels)")
    print(manual_summary)
else:
    print("refused_manual not yet filled — open full_results.csv and label OOS rows first.")

In [5]:
# Optional: inspect interactive chat logs
import sqlite3

LOG_DB = "../data/logs/chat_logs.db"
try:
    con  = sqlite3.connect(LOG_DB)
    logs = pd.read_sql("SELECT * FROM chat_logs ORDER BY id DESC", con)
    con.close()
    print(f"Chat logs: {len(logs)} entries")
    print(logs[["timestamp", "user_message", "response_ms", "error"]].head(10).to_string(index=False))
except Exception as e:
    print(f"No chat logs found ({e})")

Chat logs: 10 entries
                       timestamp                                                             user_message  response_ms error
2026-04-20T14:48:40.216528+00:00                                      how do i submit my thesis on neptun        33567  None
2026-04-20T14:43:13.993378+00:00                                      how do i submit my thesis on neptun        91598  None
2026-04-20T14:40:55.359707+00:00                                         What BSc programs are available?        92993  None
2026-04-20T08:18:30.718131+00:00                                                                       hi       106428  None
2026-04-19T16:01:57.578832+00:00                                                                       hi        85214  None
2026-04-19T15:52:47.173440+00:00                     when is the final examination in the spring semester       111064  None
2026-04-19T15:48:30.736350+00:00 whats the deadline for uploading a thesis if we are graduating in spri